# 01 - Perfilado de calidad raw

**Objetivo.** Cargar los datasets crudos desde `data/raw` y documentar estructura, columnas, tipos de datos, nulos iniciales, valores inválidos que funcionan como faltantes, rangos, cardinalidad y duplicados.

Este notebook no modifica datos. Su salida sirve como evidencia de línea de base para justificar la limpieza posterior.

## Criterio de trabajo 

- Los archivos crudos se leen con rutas relativas al repositorio.
- Los crudos no se modifican.
- Airbnb se perfila desde `airbnb_caba_normalizado_mensual.csv`, porque ya es la base normalizada prioritaria para comparación.
- Los reportes se guardan en `data/processed/reports/`.

In [1]:
from pathlib import Path
import json
import re
import unicodedata
from datetime import datetime

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except Exception:
    display = print

pd.set_option('display.max_columns', 140)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 180)


def find_repo_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for path in [start, *start.parents]:
        if (path / 'data' / 'raw').exists() and (path / 'README.md').exists():
            return path
    raise FileNotFoundError('No se encontro la raiz del repo. Ejecutar dentro del proyecto.')

REPO_ROOT = find_repo_root()
RAW_DIR = REPO_ROOT / 'data' / 'raw'
PROCESSED_DIR = REPO_ROOT / 'data' / 'processed'
REPORTS_DIR = PROCESSED_DIR / 'reports'
EXTERNAL_DIR = REPO_ROOT / 'Fuentes_de_enriquecimiento'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print('REPO_ROOT:', REPO_ROOT)
print('RAW_DIR:', RAW_DIR)
print('PROCESSED_DIR:', PROCESSED_DIR)

REPO_ROOT: C:\Users\arira\OneDrive\Documents\GitHub\TP_analitica_descriptiva_grupo1_2q2026
RAW_DIR: C:\Users\arira\OneDrive\Documents\GitHub\TP_analitica_descriptiva_grupo1_2q2026\data\raw
PROCESSED_DIR: C:\Users\arira\OneDrive\Documents\GitHub\TP_analitica_descriptiva_grupo1_2q2026\data\processed


## Carga de archivos raw

In [2]:
#set de marcadores para identificar nulos, si se encuentran mas en el analisis se agregan aca y ejecuta la limpeza nuevamente
INVALID_MARKERS = { 
    '', ' ', 'nan', 'NaN', 'NAN', 'none', 'None', 'NONE', 'null', 'NULL',
    's/d', 'S/D', 'sd', 'SD', 'sin dato', 'Sin dato', 'Sin Datos', 'SIN_DATO','sin datos',
    'no informa', 'No informa', 'no informado', 'No informado', 'n/a', 'N/A',
    '-', '--', '---', '?', '99999', '-99999', '-9999', '9999'
}

RAW_SOURCES = {
    'df_raw_meli_alq': {
        'descripcion': 'Mercado Libre - alquiler permanente',
        'paths': sorted((RAW_DIR / 'mercadolibre_inmuebles_alquiler').glob('meli_alq_*.csv')),
    },
    'df_raw_meli_alq_temp': {
        'descripcion': 'Mercado Libre - alquiler temporario',
        'paths': sorted((RAW_DIR / 'mercadolibre_inmuebles_alquiler temporario').glob('meli_alq_temp_*.csv')),
    },
    'df_raw_meli_vtas': {
        'descripcion': 'Mercado Libre - venta',
        'paths': sorted((RAW_DIR / 'mercadolibre_inmuebles_ventas').glob('meli_vtas_*.csv')),
    },
    'df_raw_argenprop': {
        'descripcion': 'Argenprop - publicaciones extraidas',
        'paths': [RAW_DIR / 'argenprop' / 'propiedades_argenprop.csv'],
    },
    'df_raw_zonaprop': {
        'descripcion': 'Zonaprop - publicaciones extraidas',
        'paths': [RAW_DIR / 'zonaprop' / 'propiedades_zonaprop.csv'],
    },
    'df_raw_airbnb_norm': {
        'descripcion': 'Airbnb CABA normalizado mensual. Base prioritaria para analisis comparable',
        'paths': [RAW_DIR / 'airbnb' / 'airbnb_caba_normalizado_mensual.csv'],
    },
}


def read_csv_safely(path):
    return pd.read_csv(path, low_memory=False)


def load_many(paths, source_name):
    existing = [Path(p) for p in paths if Path(p).exists()]
    if not existing:
        print(f'ATENCION: no se encontraron archivos para {source_name}')
        return pd.DataFrame()
    frames = []
    for path in existing:
        df = read_csv_safely(path)
        df['_archivo_origen'] = str(path.relative_to(REPO_ROOT))
        frames.append(df)
    return pd.concat(frames, ignore_index=True, sort=False)

raw_dfs = {}
for df_name, spec in RAW_SOURCES.items():
    raw_dfs[df_name] = load_many(spec['paths'], df_name)
    globals()[df_name] = raw_dfs[df_name]
    print(f"{df_name}: {raw_dfs[df_name].shape} | {spec['descripcion']}")

df_raw_meli_alq: (11479, 61) | Mercado Libre - alquiler permanente
df_raw_meli_alq_temp: (6190, 61) | Mercado Libre - alquiler temporario
df_raw_meli_vtas: (44013, 61) | Mercado Libre - venta
df_raw_argenprop: (162, 96) | Argenprop - publicaciones extraidas
df_raw_zonaprop: (980, 96) | Zonaprop - publicaciones extraidas
df_raw_airbnb_norm: (9596, 61) | Airbnb CABA normalizado mensual. Base prioritaria para analisis comparable


## Funciones de perfilado

In [3]:
COLUMN_DESCRIPTIONS = { #objetivo de ahora es entender los df con sus columnas
    'item_id': 'Identificador de publicacion en Mercado Libre.',
    'property_id': 'Identificador de publicacion o propiedad segun fuente.',
    'fuente': 'Portal o fuente de origen del registro.',
    'url': 'URL de la publicacion.',
    'search_url': 'URL de busqueda desde la cual fue capturada la publicacion.',
    'pagina_origen': 'Pagina de resultados donde aparecio la publicacion.',
    'scraped_at_utc': 'Fecha y hora de extraccion en UTC.',
    'titulo': 'Titulo publicado del aviso.',
    'descripcion': 'Descripcion textual del aviso cuando esta disponible.',
    'tipo_propiedad': 'Tipo de inmueble declarado o inferido.',
    'tipo_operacion': 'Tipo de operacion: venta, alquiler o alquiler temporal.',
    'moneda': 'Moneda del precio publicado.',
    'precio': 'Precio numerico extraido de la publicacion.',
    'precio_texto': 'Precio en formato textual original.',
    'precio_m2_min': 'Precio por metro cuadrado minimo calculado por Mercado Libre cuando hay rangos.',
    'precio_m2_max': 'Precio por metro cuadrado maximo calculado por Mercado Libre cuando hay rangos.',
    'precio_m2_calculado': 'Precio por metro cuadrado calculado por scraper en Argenprop/Zonaprop.',
    'barrio': 'Barrio declarado, inferido o capturado desde la fuente.',
    'barrio_norm': 'Barrio normalizado para analisis territorial.',
    'comuna': 'Comuna de CABA cuando esta disponible o inferida.',
    'localidad': 'Localidad declarada.',
    'provincia': 'Provincia o jurisdiccion declarada.',
    'direccion': 'Direccion capturada en Mercado Libre.',
    'direccion_completa': 'Direccion capturada en Argenprop/Zonaprop/Airbnb.',
    'calle': 'Calle extraida o inferida.',
    'altura': 'Altura aproximada o declarada.',
    'latitud': 'Latitud de la publicacion cuando la fuente la informa.',
    'longitud': 'Longitud de la publicacion cuando la fuente la informa.',
    'ambientes_min': 'Minimo de ambientes informado cuando la publicacion usa rangos.',
    'ambientes_max': 'Maximo de ambientes informado cuando la publicacion usa rangos.',
    'ambientes': 'Cantidad de ambientes.',
    'dormitorios_min': 'Minimo de dormitorios informado cuando la publicacion usa rangos.',
    'dormitorios_max': 'Maximo de dormitorios informado cuando la publicacion usa rangos.',
    'dormitorios': 'Cantidad de dormitorios.',
    'banios_min': 'Minimo de banios informado cuando la publicacion usa rangos.',
    'banios_max': 'Maximo de banios informado cuando la publicacion usa rangos.',
    'banios': 'Cantidad de banios.',
    'superficie_min_m2': 'Superficie minima en m2 informada o inferida.',
    'superficie_max_m2': 'Superficie maxima en m2 informada o inferida.',
    'superficie_total_m2': 'Superficie total en m2.',
    'superficie_cubierta_m2': 'Superficie cubierta en m2.',
    'tipo_superficie': 'Tipo de superficie usada para el dato de m2.',
    'inmobiliaria': 'Nombre de inmobiliaria o anunciante cuando esta disponible.',
    'seller_id': 'Identificador del vendedor/anfitrion cuando existe.',
    'seller_nombre': 'Nombre del anfitrion/vendedor en Airbnb.',
    'seller_superhost': 'Indicador de superhost en Airbnb.',
    'rating': 'Puntaje promedio de la publicacion en Airbnb.',
    'reviews': 'Cantidad de resenias en Airbnb.',
    'cantidad_imagenes': 'Cantidad de imagenes detectadas.',
    'imagen_principal': 'URL de imagen principal.',
    'parse_warnings': 'Advertencias del proceso de parseo.',
    'parse_ok': 'Indicador de parseo exitoso.',
    'http_status': 'Codigo HTTP de respuesta durante scraping.',
    'apto_credito': 'Indicador de apto credito.',
    'balcon': 'Indicador de balcon.',
    'balcon_o_patio': 'Indicador de balcon o patio en Airbnb.',
    'terraza': 'Indicador de terraza.',
    'pileta': 'Indicador de pileta.',
    'parrilla': 'Indicador de parrilla.',
    'cochera_mencionada': 'Indicador de cochera mencionada en Mercado Libre.',
    'cochera': 'Indicador de cochera en Airbnb/otros portales.',
    'amenities_mencionadas': 'Indicador de amenities mencionadas en Mercado Libre.',
    'precio_sospechoso': 'Indicador de precio sospechoso en Airbnb normalizado.',
    'unidad_precio': 'Unidad de referencia del precio Airbnb.',
    'precio_regimen': 'Regimen de precio Airbnb: por noche, mensual, etc.',
}


def example_value(series):
    sample = series.dropna().head(1)
    if sample.empty:
        return np.nan
    text = str(sample.iloc[0])
    return text[:120] + ('...' if len(text) > 120 else '')


def column_dictionary(df, dataset_name):
    rows = []
    for col in df.columns:
        rows.append({
            'dataset': dataset_name,
            'columna': col,
            'tipo_dato': str(df[col].dtype),
            'descripcion': COLUMN_DESCRIPTIONS.get(col, 'Pendiente de documentar segun significado en la fuente.'),
            'ejemplo': example_value(df[col]),
        })
    return pd.DataFrame(rows)


def invalid_like_missing_count(series):
    if not (pd.api.types.is_object_dtype(series) or pd.api.types.is_string_dtype(series)):
        return 0
    normalized = series.astype('string').str.strip()
    return int(normalized.isin(INVALID_MARKERS).sum())


def usability_comment(null_pct, invalid_pct, n_labels, dtype):
    missing_pct = max(null_pct, invalid_pct)
    if missing_pct >= 80:
        return 'baja: faltantes muy altos; usar solo como referencia secundaria'
    if missing_pct >= 40:
        return 'media-baja: requiere justificacion antes de usar en KPIs'
    if missing_pct >= 15:
        return 'media: usable con controles y posible segmentacion'
    if n_labels > 300 and dtype == 'categorica':
        return 'media: alta cardinalidad; revisar si es identificador/texto libre'
    return 'alta: utilizable para analisis inicial'


def quality_profile(df, dataset_name):
    rows = []
    n = len(df)
    for col in df.columns:
        s = df[col]
        null_count = int(s.isna().sum())
        invalid_count = invalid_like_missing_count(s)
        is_num = pd.api.types.is_numeric_dtype(s)
        dtype_group = 'cuantitativa' if is_num else 'categorica/texto/fecha'
        numeric_min = s.min(skipna=True) if is_num and s.notna().any() else np.nan
        numeric_max = s.max(skipna=True) if is_num and s.notna().any() else np.nan
        n_labels = int(s.nunique(dropna=True)) if not is_num else np.nan
        null_pct = round(null_count / n * 100, 2) if n else np.nan
        invalid_pct = round(invalid_count / n * 100, 2) if n else np.nan
        rows.append({
            'dataset': dataset_name,
            'columna': col,
            'tipo_dato': str(s.dtype),
            'tipo_variable': dtype_group,
            'n_filas': n,
            'nulos_iniciales': null_count,
            'pct_nulos_iniciales': null_pct,
            'invalidos_tipo_faltante': invalid_count,
            'pct_invalidos_tipo_faltante': invalid_pct,
            'min_cuantitativa': numeric_min,
            'max_cuantitativa': numeric_max,
            'n_labels_categorica': n_labels,
            'comentario_usabilidad_calidad': usability_comment(null_pct, invalid_pct, 0 if pd.isna(n_labels) else n_labels, 'cuantitativa' if is_num else 'categorica'),
        })
    return pd.DataFrame(rows)


def dataset_inventory(dfs):
    rows = []
    for name, df in dfs.items():
        scraped_col = 'scraped_at_utc' if 'scraped_at_utc' in df.columns else None
        scraped_min = scraped_max = np.nan
        if scraped_col:
            dates = pd.to_datetime(df[scraped_col], errors='coerce', utc=True)
            if dates.notna().any():
                scraped_min = dates.min()
                scraped_max = dates.max()
        rows.append({
            'dataframe': name,
            'filas': len(df),
            'columnas': df.shape[1],
            'memoria_mb': round(df.memory_usage(deep=True).sum() / 1024**2, 2),
            'fecha_extraccion_min': scraped_min,
            'fecha_extraccion_max': scraped_max,
            'archivos_origen': df['_archivo_origen'].nunique() if '_archivo_origen' in df.columns else np.nan,
        })
    return pd.DataFrame(rows)

## Inventario general de datasets

In [4]:
inventario_raw = dataset_inventory(raw_dfs)
display(inventario_raw) #resumen de conformado de datasets y extracciones
inventario_raw.to_csv(REPORTS_DIR / 'inventario_raw.csv', index=False)

,dataframe,filas,columnas,memoria_mb,fecha_extraccion_min,fecha_extraccion_max,archivos_origen
0,df_raw_meli_alq,11479,61,26.90,2026-09-03 16:45:14.494669+00:00,2026-09-05 20:06:33.128720+00:00,2
1,df_raw_meli_alq_temp,6190,61,15.01,2026-09-03 17:22:27.502870+00:00,2026-09-05 20:52:13.036557+00:00,2
2,df_raw_meli_vtas,44013,61,103.65,2026-08-15 15:28:55.553461+00:00,2026-09-03 16:42:08.328566+00:00,2
3,df_raw_argenprop,162,96,0.63,2026-08-15 23:56:14.395650+00:00,2026-09-03 15:09:36.365547+00:00,1
4,df_raw_zonaprop,980,96,5.76,2026-08-17 20:48:48.072806+00:00,2026-09-03 15:12:53.379325+00:00,1
5,df_raw_airbnb_norm,9596,61,33.83,2026-09-05 16:19:29+00:00,2026-09-06 05:39:56+00:00,1


## Diccionario inicial de columnas

Tabla con columna, tipo de dato, descripción inicial y ejemplo, a referenciar siempre que se necesite entender una columna. Las descripciones pendientes deben completarse si la variable se vuelve central para el análisis.

In [5]:
diccionario_raw_inicial = pd.concat([column_dictionary(df, name) for name, df in raw_dfs.items()], ignore_index=True)
display(diccionario_raw_inicial.head(100))
diccionario_raw_inicial.to_csv(PROCESSED_DIR / 'diccionario_raw_inicial.csv', index=False)

,dataset,columna,tipo_dato,descripcion,ejemplo
0,df_raw_meli_alq,item_id,str,Identificador de publicacion en Mercado Libre.,MLA2052442697
1,df_raw_meli_alq,fuente,str,Portal o fuente de origen del registro.,Mercado Libre
2,df_raw_meli_alq,url,str,URL de la publicacion.,https://departamento.mercadolibre.com.ar/MLA-2...
3,df_raw_meli_alq,search_url,str,URL de busqueda desde la cual fue capturada la...,https://inmuebles.mercadolibre.com.ar/departam...
4,df_raw_meli_alq,pagina_origen,int64,Pagina de resultados donde aparecio la publica...,1
5,df_raw_meli_alq,scraped_at_utc,str,Fecha y hora de extraccion en UTC.,2026-09-03T16:45:14.494669+00:00
6,df_raw_meli_alq,titulo,str,Titulo publicado del aviso.,Departamento En Alquiler De 4 Ambientes Con De...
7,df_raw_meli_alq,tipo_propiedad,str,Tipo de inmueble declarado o inferido.,departamento
8,df_raw_meli_alq,tipo_operacion,str,"Tipo de operacion: venta, alquiler o alquiler ...",alquiler
9,df_raw_meli_alq,moneda,str,Moneda del precio publicado.,USD


## Columnas, tipos y ejemplos por dataset

In [6]:
for name, df in raw_dfs.items():
    print('\n' + '=' * 100)
    print(name, df.shape)
    display(pd.DataFrame({
        'columna': df.columns,
        'tipo_dato': [str(df[c].dtype) for c in df.columns],
        'ejemplo': [example_value(df[c]) for c in df.columns],
    }))


df_raw_meli_alq (11479, 61)


,columna,tipo_dato,ejemplo
0,item_id,str,MLA2052442697
1,fuente,str,Mercado Libre
2,url,str,https://departamento.mercadolibre.com.ar/MLA-2...
3,search_url,str,https://inmuebles.mercadolibre.com.ar/departam...
4,pagina_origen,int64,1
5,scraped_at_utc,str,2026-09-03T16:45:14.494669+00:00
6,titulo,str,Departamento En Alquiler De 4 Ambientes Con De...
7,tipo_propiedad,str,departamento
8,tipo_operacion,str,alquiler
9,moneda,str,USD



df_raw_meli_alq_temp (6190, 61)


,columna,tipo_dato,ejemplo
0,item_id,str,MLA2698335478
1,fuente,str,Mercado Libre
2,url,str,https://departamento.mercadolibre.com.ar/MLA-2...
3,search_url,str,https://inmuebles.mercadolibre.com.ar/departam...
4,pagina_origen,int64,1
5,scraped_at_utc,str,2026-09-03T17:22:27.502870+00:00
6,titulo,str,Departamento Alquiler Temporario 3 Ambientes A...
7,tipo_propiedad,str,departamento
8,tipo_operacion,str,alquiler temporal
9,moneda,str,USD



df_raw_meli_vtas (44013, 61)


...

,columna,tipo_dato,ejemplo
0,property_id,str,e197e9f1c643db6d77d3
1,fuente,str,Argenprop
2,url,str,https://www.argenprop.com/departamento-en-vent...
3,scraped_at_utc,str,2026-08-15T23:56:14.395650+00:00
4,search_url,str,https://www.argenprop.com/departamentos/venta/...
5,pagina_origen,int64,1
6,http_status,int64,200
7,parse_ok,int64,1
8,parse_warnings,str,modo_listado_sin_ficha_detallada
9,titulo,str,TORRE BELLINI! Piso alto de revista! Balcón co...



df_raw_zonaprop (980, 96)


,columna,tipo_dato,ejemplo
0,property_id,str,174f7e930591b69560cf
1,fuente,str,Zonaprop
2,url,str,https://www.zonaprop.com.ar/propiedades/clasif...
3,scraped_at_utc,str,2026-08-17T20:48:48.072806+00:00
4,search_url,str,https://www.zonaprop.com.ar/departamentos-vent...
5,pagina_origen,int64,1
6,http_status,int64,200
7,parse_ok,int64,1
8,parse_warnings,str,modo_listado_sin_ficha_detallada
9,titulo,str,Una nueva forma de vivir el sur porteño. Desar...



df_raw_airbnb_norm (9596, 61)


,columna,tipo_dato,ejemplo
0,property_id,int64,1040554442322631694
1,fuente,str,Airbnb
2,tipo_operacion,str,alquiler temporal
3,unidad_precio,str,por_noche_2_huespedes_5_noches
4,url,str,https://www.airbnb.com.ar/rooms/10405544423226...
5,scraped_at_utc,str,2026-09-05T17:38:48Z
6,titulo,str,La mejor ubicación en Palermo - La Rural
7,descripcion,str,Cómodo y céntrico departamento en el corazón d...
8,tipo_propiedad,str,Departamento con servicios incluidos entero
9,room_type,str,Entire home/apt


## Perfil de calidad por columna

In [7]:
perfil_calidad_raw = pd.concat([quality_profile(df, name) for name, df in raw_dfs.items()], ignore_index=True)
perfil_calidad_raw = perfil_calidad_raw.sort_values(['dataset', 'pct_nulos_iniciales', 'pct_invalidos_tipo_faltante'], ascending=[True, False, False])
display(perfil_calidad_raw.head(160))
perfil_calidad_raw.to_csv(PROCESSED_DIR / 'perfil_calidad_raw.csv', index=False)
#analisis rapido de la calidad

,dataset,columna,tipo_dato,tipo_variable,n_filas,nulos_iniciales,pct_nulos_iniciales,invalidos_tipo_faltante,pct_invalidos_tipo_faltante,min_cuantitativa,max_cuantitativa,n_labels_categorica,comentario_usabilidad_calidad
408,df_raw_airbnb_norm,rating,float64,cuantitativa,9596,1522,15.86,0,0.0,1.0,5.0,NaN,media: usable con controles y posible segmenta...
409,df_raw_airbnb_norm,reviews,float64,cuantitativa,9596,1522,15.86,0,0.0,1.0,1140.0,NaN,media: usable con controles y posible segmenta...
403,df_raw_airbnb_norm,camas,float64,cuantitativa,9596,189,1.97,0,0.0,1.0,40.0,NaN,alta: utilizable para analisis inicial
394,df_raw_airbnb_norm,barrio,str,categorica/texto/fecha,9596,77,0.80,0,0.0,NaN,NaN,45.0,alta: utilizable para analisis inicial
395,df_raw_airbnb_norm,barrio_norm,str,categorica/texto/fecha,9596,77,0.80,0,0.0,NaN,NaN,45.0,alta: utilizable para analisis inicial
...,...,...,...,...,...,...,...,...,...,...,...,...,...
276,df_raw_argenprop,atributos_extra_json,str,categorica/texto/fecha,162,0,0.00,0,0.0,NaN,NaN,162.0,alta: utilizable para analisis inicial
278,df_raw_argenprop,_archivo_origen,str,categorica/texto/fecha,162,0,0.00,0,0.0,NaN,NaN,1.0,alta: utilizable para analisis inicial
30,df_raw_meli_alq,posesion,float64,cuantitativa,11479,11479,100.00,0,0.0,NaN,NaN,NaN,baja: faltantes muy altos; usar solo como refe...
31,df_raw_meli_alq,unidades_disponibles,float64,cuantitativa,11479,11479,100.00,0,0.0,NaN,NaN,NaN,baja: faltantes muy altos; usar solo como refe...


## Variables con mayores problemas de completitud

In [8]:
problemas_completitud = (
    perfil_calidad_raw
    .query('pct_nulos_iniciales >= 20 or pct_invalidos_tipo_faltante >= 5')
    .sort_values(['dataset', 'pct_nulos_iniciales', 'pct_invalidos_tipo_faltante'], ascending=[True, False, False])
)
display(problemas_completitud)
problemas_completitud.to_csv(REPORTS_DIR / 'problemas_completitud_raw.csv', index=False)

,dataset,columna,tipo_dato,tipo_variable,n_filas,nulos_iniciales,pct_nulos_iniciales,invalidos_tipo_faltante,pct_invalidos_tipo_faltante,min_cuantitativa,max_cuantitativa,n_labels_categorica,comentario_usabilidad_calidad
193,df_raw_argenprop,descripcion,float64,cuantitativa,162,162,100.00,0,0.0,NaN,NaN,NaN,baja: faltantes muy altos; usar solo como refe...
196,df_raw_argenprop,estado_publicacion,float64,cuantitativa,162,162,100.00,0,0.0,NaN,NaN,NaN,baja: faltantes muy altos; usar solo como refe...
197,df_raw_argenprop,fecha_publicacion,float64,cuantitativa,162,162,100.00,0,0.0,NaN,NaN,NaN,baja: faltantes muy altos; usar solo como refe...
198,df_raw_argenprop,inmobiliaria,float64,cuantitativa,162,162,100.00,0,0.0,NaN,NaN,NaN,baja: faltantes muy altos; usar solo como refe...
199,df_raw_argenprop,seller_id,float64,cuantitativa,162,162,100.00,0,0.0,NaN,NaN,NaN,baja: faltantes muy altos; usar solo como refe...
...,...,...,...,...,...,...,...,...,...,...,...,...,...
320,df_raw_zonaprop,cocheras,float64,cuantitativa,980,916,93.47,0,0.0,1.0,200.0,NaN,baja: faltantes muy altos; usar solo como refe...
306,df_raw_zonaprop,piso,str,categorica/texto/fecha,980,613,62.55,0,0.0,NaN,NaN,5.0,media-baja: requiere justificacion antes de us...
300,df_raw_zonaprop,expensas,float64,cuantitativa,980,326,33.27,0,0.0,3.0,672.0,NaN,media: usable con controles y posible segmenta...
301,df_raw_zonaprop,expensas_texto,str,categorica/texto/fecha,980,326,33.27,0,0.0,NaN,NaN,182.0,media: usable con controles y posible segmenta...


En principio las columnas con 100% no aportan usabilidad, aunque no se descarta completarlas con fuentes externas, ingenieria de atributos, etc. Si la columna entera esta vacia, lo mas probable se deba a problemas en la carga y el scrapping; pendiente de analisis en los futuros notebooks y la entrega 3 donde se trabaje con texto. 

## Rangos de variables cuantitativas

Este control va a permitir detectar valores extremos, errores de escala y columnas numéricas que en realidad son flags o identificadores. El tratamiento de outliers queda separado en el notebook 04.

In [9]:
rangos = []
for name, df in raw_dfs.items():
    num_cols = df.select_dtypes(include='number').columns
    if len(num_cols) == 0:
        continue
    tmp = df[num_cols].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).T.reset_index()
    tmp = tmp.rename(columns={'index': 'columna'})
    tmp.insert(0, 'dataset', name)
    rangos.append(tmp)

rangos_cuantitativos_raw = pd.concat(rangos, ignore_index=True) if rangos else pd.DataFrame()
display(rangos_cuantitativos_raw)
rangos_cuantitativos_raw.to_csv(REPORTS_DIR / 'rangos_cuantitativos_raw.csv', index=False)

,dataset,columna,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max
0,df_raw_meli_alq,pagina_origen,11479.0,8.589337e+00,9.126467e+00,1.0,1.0,1.0,2.00,5.000000e+00,1.200000e+01,3.100000e+01,4.000000e+01,4.200000e+01
1,df_raw_meli_alq,precio,11479.0,6.479774e+05,1.161667e+06,10.0,500.0,750.0,4000.00,6.500000e+05,8.800000e+05,1.500000e+06,2.300000e+06,1.111111e+08
2,df_raw_meli_alq,desde,11479.0,6.969248e-04,2.639128e-02,0.0,0.0,0.0,0.00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,1.000000e+00
3,df_raw_meli_alq,ambientes_min,11140.0,2.187882e+00,1.313836e+00,1.0,1.0,1.0,1.00,2.000000e+00,3.000000e+00,4.000000e+00,5.000000e+00,7.100000e+01
4,df_raw_meli_alq,ambientes_max,11140.0,2.187882e+00,1.313836e+00,1.0,1.0,1.0,1.00,2.000000e+00,3.000000e+00,4.000000e+00,5.000000e+00,7.100000e+01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
270,df_raw_airbnb_norm,capacidad,9596.0,2.945185e+00,1.458231e+00,2.0,2.0,2.0,2.00,2.000000e+00,4.000000e+00,5.000000e+00,8.000000e+00,2.600000e+01
271,df_raw_airbnb_norm,seller_id,9596.0,3.068990e+16,2.272244e+17,13426.0,416887.1,2120738.0,36668835.00,1.604368e+08,4.692825e+08,6.971964e+08,1.706666e+18,1.767139e+18
272,df_raw_airbnb_norm,rating,8074.0,4.736255e+00,4.271341e-01,1.0,3.0,4.0,4.68,4.840000e+00,4.970000e+00,5.000000e+00,5.000000e+00,5.000000e+00
273,df_raw_airbnb_norm,reviews,8074.0,3.814144e+01,5.276317e+01,1.0,1.0,1.0,6.00,1.900000e+01,5.100000e+01,1.323500e+02,2.380000e+02,1.140000e+03


## Cardinalidad de variables categóricas

In [10]:
cardinalidades = []
for name, df in raw_dfs.items():
    cat_cols = df.select_dtypes(include=['object', 'string', 'category', 'bool']).columns
    for col in cat_cols:
        vc = df[col].astype('string').str.strip().value_counts(dropna=False).head(10)
        cardinalidades.append({
            'dataset': name,
            'columna': col,
            'n_labels_incluyendo_na': int(df[col].nunique(dropna=False)),
            'top_10_labels': ' | '.join([f'{idx}: {val}' for idx, val in vc.items()])[:1000],
        })
cardinalidad_categoricas_raw = pd.DataFrame(cardinalidades).sort_values(['dataset', 'n_labels_incluyendo_na'], ascending=[True, False])
display(cardinalidad_categoricas_raw.head(140))
cardinalidad_categoricas_raw.to_csv(REPORTS_DIR / 'cardinalidad_categoricas_raw.csv', index=False)

,dataset,columna,n_labels_incluyendo_na,top_10_labels
118,df_raw_airbnb_norm,url,9596,https://www.airbnb.com.ar/rooms/10405544423226...
119,df_raw_airbnb_norm,scraped_at_utc,9596,2026-09-05T17:38:48Z: 1 | 2026-09-06T01:02:39Z...
136,df_raw_airbnb_norm,imagen_principal,9399,https://a0.muscache.com/im/pictures/b3dd86a1-e...
120,df_raw_airbnb_norm,titulo,9201,Departamento en Buenos Aires: 25 | Departament...
121,df_raw_airbnb_norm,descripcion,8801,Disfrutá de la sencillez de este alojamiento t...
...,...,...,...,...
59,df_raw_meli_vtas,moneda,2,USD: 44010 | ARS: 3
65,df_raw_meli_vtas,provincia,2,Capital Federal: 44009 | <NA>: 4
75,df_raw_meli_vtas,imagen_extension,2,webp: 44012 | jpg: 1
76,df_raw_meli_vtas,_archivo_origen,2,data\raw\mercadolibre_inmuebles_ventas\meli_vt...


Revisar este reporte permite revisar rapidamente si las labels efectivamente corresponden a categorizaciones pertinentes o si abundan columnas categoricas de alta cardianlidad (fechas, ids, etc). Su tratamiento se decide en ingenieria de variables (buckets). 

## Duplicados iniciales

In [11]:
keys_candidates = ['item_id', 'property_id', 'url']
rows = []
for name, df in raw_dfs.items():
    row = {'dataset': name, 'filas': len(df), 'duplicados_exactos': int(df.duplicated().sum())}
    for key in keys_candidates:
        if key in df.columns:
            row[f'duplicados_por_{key}'] = int(df.duplicated(subset=[key]).sum())
            row[f'n_unicos_{key}'] = int(df[key].nunique(dropna=True))
    rows.append(row)
duplicados_raw = pd.DataFrame(rows)
display(duplicados_raw)
duplicados_raw.to_csv(REPORTS_DIR / 'duplicados_raw.csv', index=False)

,dataset,filas,duplicados_exactos,duplicados_por_item_id,n_unicos_item_id,duplicados_por_url,n_unicos_url,duplicados_por_property_id,n_unicos_property_id
0,df_raw_meli_alq,11479,0,1602.0,9877.0,1583,9896,NaN,NaN
1,df_raw_meli_alq_temp,6190,0,1696.0,4494.0,1690,4500,NaN,NaN
2,df_raw_meli_vtas,44013,0,1235.0,42778.0,1188,42825,NaN,NaN
3,df_raw_argenprop,162,0,NaN,NaN,0,162,0.0,162.0
4,df_raw_zonaprop,980,0,NaN,NaN,0,980,0.0,980.0
5,df_raw_airbnb_norm,9596,0,NaN,NaN,0,9596,0.0,9596.0


Existen 0 duplicados exactos, aunque parece repetirse propiedades. Se debera corroboraar que efectivamente se tratan de la misma propiedad y unificar los datos, aunque dos publicaciones diferentes pueden incluir diferentes precios. Se decidira un tratamiento adecuado de acuerdo a los alcances del analisis. 

## Cierre del perfilado raw

Hallazgos a trasladar al notebook 02:

- Las fuentes deben normalizarse por separado antes de consolidar, porque tienen estructuras y campos propios. Se mantienen DFs separados por estas etapas para evitar mezclar estructuras, generar nulos artificialmente, etc.
- Los faltantes estructurales entre portales no deben confundirse con problemas de calidad. Se deben analizar los faltantes segun su proporcion en cada columna y aleatoriedad/ falta de. 
- Las variables de precio, superficie, moneda, barrio/comuna y operación son críticas para el análisis posterior.
- Los valores extremos se identifican preliminarmente en esta etapa, pero se tratan con flags específicos en el notebook 04, definiendo analisis y decision. 